# Fields to update
1. MB_ID (Done)
2. Country
3. Sub_genre
4. Homepage
5. Bandcamp page

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe


In [4]:
# Notion imports
from Notion.reader import get_artists_db
from Notion.writer import build_property, update_artist
from Notion.database import myNotionDatabases
from Data.artists import Artist
from Data.tags import clean_tags
from Datasources.Last_Fm import get_artist_tags
# MusicBrainz imports
from Datasources.MusicBrainz import search_artist,get_artist_info, load_cache, save_cache
from tqdm import tqdm
import random
import copy

In [5]:
notion_response = get_artists_db()
#artists_db = []
db = myNotionDatabases()
Updated_notion_artist_database = {}

for artist in notion_response:
    extracted_artist = db.extract_artist(artist)
    #artists_db.append(extracted_artist)
    key = extracted_artist['artist_name'].lower().strip()
    Updated_notion_artist_database[key] = Artist(
        name=extracted_artist['artist_name'],
        mbid=extracted_artist['mb_id'],
        country=extracted_artist['country'],
        tags=extracted_artist.get('subgenre', []),
        website_url=extracted_artist.get('website'),
        bandcamp_url=extracted_artist.get('bandcamp_url'),
        wikidata_url=extracted_artist.get('wikidata_url'),
        bandsintown_url=extracted_artist.get('bandsintown_url'),
        page_id=extracted_artist.get('id')
    )


Original_notion_artist_database = copy.deepcopy(Updated_notion_artist_database)

In [6]:
#for artist in random.sample(sorted(notion_artist_database.values(), key=lambda a: a.name), min(10, len(notion_artist_database))): #Selecting X random artists for testing purposes
no_matches = []
for artist in tqdm(Updated_notion_artist_database.values()):
    if artist.mbid is None:
        try:
            mb_id, match_result = search_artist(artist_name=artist.name.strip(), confidence_threshold=85)
            if mb_id is None:
                #this will force an update and send the result to the notion database. 
                # I can then make a view which filters for them.
                artist.mbid = f"No match found for |{artist.name}| with the reason: {match_result}"
            else:
                artist.mbid = mb_id
        except Exception as e:
            print(f"Error processing {artist.name}: {e}")

    if artist.mbid is not None and not artist.mbid.startswith("No match found"):
        try:
            mb_artist_info = get_artist_info(artist.mbid)
            if artist.country is None:
                artist.country = mb_artist_info.get("country")
            if artist.website_url is None:
                artist.website_url = mb_artist_info.get("homepage")
            if artist.bandcamp_url is None:
                artist.bandcamp_url = mb_artist_info.get("bandcamp")
            if artist.wikidata_url is None:
                artist.wikidata_url = mb_artist_info.get("wikidata")
            if artist.bandsintown_url is None:
                artist.bandsintown_url = mb_artist_info.get("Bandsintown")
            if not artist.tags:
                artist.tags = mb_artist_info.get("tags", [])
        except Exception as e:
            print(f"Error fetching info for {artist.name}: {e}")


100%|██████████| 144/144 [00:11<00:00, 12.90it/s]


In [7]:
for error in no_matches:
    print(error)
print(f"Number of missing matches = {len(no_matches)}")
no_exact_match = len([x for x in no_matches if "No Exact Match" in x])
print(f"No Exact Match Issues = {no_exact_match}")
multi_exact_match = len([x for x in no_matches if "Multiple Exact Matches" in x])
print(f"Multiple Exact Match Issues = {multi_exact_match}")

Number of missing matches = 0
No Exact Match Issues = 0
Multiple Exact Match Issues = 0


In [8]:
#Get the Last.fm info for the artist.
last_fm_artists = []
for artist in Updated_notion_artist_database.values():
    if artist.mbid is not None:
        last_fm_artists.append(artist)

In [9]:
for artist in tqdm(last_fm_artists):
    #print(artist.name)
    last_fm_response = get_artist_tags(mb_id=artist.mbid)
    artist.add_tags(last_fm_response)

100%|██████████| 144/144 [00:20<00:00,  7.16it/s]


In [10]:
for artist in tqdm(last_fm_artists):
    cleaned_tags = clean_tags(artist.tags)
    sorted_tags = sorted(cleaned_tags.items(), key=lambda x: x[1], reverse=True)
    Top_x = sorted_tags[:3]
    declared_subgenres = []

    for tag in Top_x:
        declared_subgenres.append(tag)
    artist.tags = Top_x

100%|██████████| 144/144 [00:00<00:00, 71988.05it/s]


In [ ]:
for key, original in Original_notion_artist_database.items():
    updated_data = Updated_notion_artist_database[key]
    updates = {}
    page_id = original.page_id
    if 'No match found' in updated_data.mbid:
        updates["MB_ID"] = ("rich_text", updated_data.mbid)
    else:
        if original.mbid is None and updated_data.mbid != None:
            updates["MB_ID"] = ("rich_text", updated_data.mbid)
        if original.country is None and updated_data.country != None:
            updates["Country"] = ("rich_text", updated_data.country)
        if original.bandcamp_url is None and updated_data.bandcamp_url != None:
            updates["Bandcamp url"] = ("url", updated_data.bandcamp_url)
        if original.website_url is None and updated_data.website_url != None:
            updates["Website"] = ("url", updated_data.website_url)
        if original.wikidata_url is None and updated_data.wikidata_url != None:
            updates["Wikidata_URL"] = ("url", updated_data.wikidata_url)
        if original.bandsintown_url is None and updated_data.bandsintown_url != None:
            updates["Bandsintown_URL"] = ("url", updated_data.bandsintown_url)
    
    if len(updates) != 0:
        update_artist(page_id=page_id, updates=updates, dry_run=True)#This line acts as a printout
        update_artist(page_id=page_id, updates=updates, dry_run=False)

DRY RUN — 36b6e950-cac6-8046-b986-e1d94fd20b45:
  MB_ID: {'rich_text': [{'text': {'content': "No match found for |Ice Nine Nails| with the reason: No Match — best fuzzy was 'nine inch nails' (69%)"}}]}
DRY RUN — 36b6e950-cac6-805c-916f-fd93bd4d3578:
  MB_ID: {'rich_text': [{'text': {'content': "No match found for |Devil Driver| with the reason: No Match — best fuzzy was 'hard driver' (70%)"}}]}
DRY RUN — 3526e950-cac6-80d5-bbd6-c6fe0e343ab9:
  MB_ID: {'rich_text': [{'text': {'content': "No match found for |Ghostkid| with the reason: No Match — best fuzzy was 'ghøstkid' (88%)"}}]}
Updating Wikidata URL for The Amazons to https://www.wikidata.org/wiki/Q28754328
DRY RUN — 3486e950-cac6-80e2-9080-e79111c9bd23:
  Country: {'rich_text': [{'text': {'content': 'United Kingdom'}}]}
  Website: {'url': 'https://theamazons.co.uk/'}
  Wikidata_URL: {'url': 'https://www.wikidata.org/wiki/Q28754328'}
DRY RUN — 3476e950-cac6-807c-962c-d076a63a7033:
  MB_ID: {'rich_text': [{'text': {'content': "No matc